# Markierungs-Export für radinfra.de

Veröffentlicht jedes erkannte Fahrrad-Symbol auf der Fahrbahn als GeoJSON, dazu README
und Statistikgrafik. **Läuft unbeaufsichtigt in Produktion:** donnerstags 01:00 UTC über
`scripts/run_mapillary_notebooks.sh mk`; danach baut
[`radinfra_2_pmtiles.ipynb`](radinfra_2_pmtiles.ipynb) die Vector Tiles.

Im Aufbau identisch zum Zwilling der Verkehrszeichen-Kampagne
([`../cycleway_complete_campaign/radinfra_1_export.ipynb`](../cycleway_complete_campaign/radinfra_1_export.ipynb)).
Die Logik liegt in [`../cw_campaign.py`](../cw_campaign.py).

**Erzeugt in `mk_output/`**

| Datei | |
| --- | --- |
| `mapillary_markings_bicycle_latest.geojson.gz` | der eigentliche Export |
| `markings_by_month.svg` | Statistikgrafik für die README |
| `README.md` | Beschreibung des Ordners |

> **Kein `sync_features` hier.** Anders als die `maproulette_tasks`-Notebooks holt dieses
> sich die Daten *nicht* von data.vizsim.de: auf dem Server entstehen die Parquets in
> Schritt 1 derselben Pipeline und werden erst danach hochgeladen. Ein Sync würde sie durch
> die Vorwochen-Fassung ersetzen.

## Einstellungen

In [ ]:
# Kein %autoreload hier - anders als maproulette_tasks läuft dieses Notebook unbeaufsichtigt
# per nbconvert auf dem Server.
from pathlib import Path

import pandas as pd

# cw_campaign.py liegt eine Ebene höher, weil beide Cycleway-Kampagnen es nutzen.
# Das ".." funktioniert, weil Jupyter und nbconvert das Arbeitsverzeichnis auf das
# Notebook-Verzeichnis setzen — dieselbe Annahme wie bei ../../output/ und ../utils/.
import sys

sys.path.insert(0, "..")
import cw_campaign as cw

ordner_punkte = Path("../../output")
pfad_metadaten = ordner_punkte / "ml-mf_metadata.json"
pfad_grenze = Path("../utils/OSMB-germany.geojson.gz")
ordner_ausgabe = Path("mk_output")

# Für radinfra.de wird weit zurückgeschaut: hier zählt jede Erkennung, die lange
# genug sichtbar war, nicht nur die aktuellen.
zuletzt_gesehen_nach = "2023-01-01"
# Schutz gegen unplausible Datumsangaben in den Rohdaten.
erstmals_gesehen_nach = "2000-01-01"
# Mindestabstand zwischen erster und letzter Sichtung, in Tagen.
mindest_tage = 180

# Nur die Spalten lesen, die gebraucht werden - die map-feature-points-Parquets
# enthalten alle Feature-Klassen und sind entsprechend groß.
spalten = ["id", "value", "first_seen_at", "last_seen_at", "geometry"]

ordner_ausgabe.mkdir(parents=True, exist_ok=True)
print("Markierungen:", ", ".join(cw.MARKIERUNGEN))

## 1 · Markierungen einlesen

Gefiltert wird **beim Lesen**, je Datei. Das spart RAM — und es bestimmt den
Zeilenindex, der beim Export zur Top-Level-`id` der GeoJSON-Features wird. Wer erst
nach dem Zusammenfügen filtert, bekommt dieselben Zeilen mit anderen ids, und
tippecanoe trägt die in die Vector Tiles.

In [ ]:
# Sollwert aus dem Tile-Cache; ohne Tile-Cache (nur gespiegelte Parquets) None.
erwartet = cw.count_expected_states("../../prep/tile_cache/DE-*_tiles.json")
print("Tile-Cache kennt", erwartet, "Bundesländer")

marks = cw.load_features(
    ordner_punkte,
    prefix=cw.PREFIX_MARKIERUNGEN,
    values=cw.MARKIERUNGEN,
    columns=spalten,
    expect_files=erwartet,
    seen_after=zuletzt_gesehen_nach,
    first_seen_after=erstmals_gesehen_nach,
)
marks.head()

In [ ]:
ml_data_from, processed_date, bundeslaender = cw.read_dataset_metadata(pfad_metadaten)
print("ml_data_from: ", ml_data_from)
print("processed_date:", processed_date)
print("Bundesländer:  ", len(bundeslaender))

## 2 · Filter

In [ ]:
# Die Parquets sind nach Zoom-14-Kacheln geschnitten, Randkacheln ragen ins Ausland.
marks = cw.clip_to_boundary(marks, cw.load_boundary(pfad_grenze))

In [ ]:
# Mindestens zwei Sichtungen mit Abstand - eine einmalige Erkennung kann eine
# Fehlerkennung oder eine temporäre Markierung sein.
marks = cw.filter_by_days_seen(marks, mindest_tage)

## 3 · Export schreiben

In [ ]:
export = cw.add_marking_label(marks)[cw.EXPORT_SPALTEN_MARKIERUNGEN]
pfad_geojson = ordner_ausgabe / "mapillary_markings_bicycle_latest.geojson.gz"

cw.write_geojson_gz(export, pfad_geojson)
cw.assert_ids_are_strings(pfad_geojson)

## 4 · Statistikgrafik

Reine Darstellung, deshalb bewusst im Notebook geblieben und nicht im Modul.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="notebook", rc={
    "axes.facecolor": "#e4e4ed",
    "figure.facecolor": "#DADADF",
    "grid.linestyle": ":",
    "grid.alpha": 0.7,
})

monat = pd.to_datetime(export["last_seen_at"]).dt.to_period("M").dt.to_timestamp()
pro_monat = export.groupby(monat).size().sort_index()
# Months without a detection keep their slot, so the time axis stays linear - as in the ts chart.
pro_monat = pro_monat.reindex(pd.date_range(pro_monat.index.min(), pro_monat.index.max(), freq="MS"), fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
pro_monat.plot(kind="bar", ax=ax, color=plt.get_cmap("tab10")(0))

ax.set_xticks(range(len(pro_monat.index)))
ax.set_xticklabels([d.strftime("%Y-%m") for d in pro_monat.index], rotation=45, ha="right")
ax.tick_params(axis="x", labelsize=9)
ax.ticklabel_format(style="plain", axis="y")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}"))
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
ax.tick_params(axis="y", labelsize=10)
ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Year-Month")
ax.set_ylabel("Number of markings")
ax.set_title("Detected road markings per month\nBicycle symbol")

# Totals above the bars.
for i, summe in enumerate(pro_monat.to_numpy()):
    if summe > 0:
        ax.text(i, summe + max(pro_monat) * 0.01, str(int(summe)), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(ordner_ausgabe / "markings_by_month.svg", bbox_inches="tight")
plt.show()

## 5 · README schreiben

In [ ]:
readme = cw.build_readme_markierungen(
    anzahl=len(export),
    stand=cw.dataset_stand([ml_data_from, processed_date]),
    seit=zuletzt_gesehen_nach,
    min_days=mindest_tage,
)
(ordner_ausgabe / "README.md").write_text(readme, encoding="utf-8")
print(readme[:420])